# Laboratorium 9 - Sieci Rekurencyjne

Na tym laboratorium zapoznamy się z sieciami rekurencyjnymi - jedną z architektur dedykowanych danym sekwencyjnym. Dla danych sekwencyjnych, jednym z naturalnych podejść do modelowania jest założenie, że przetwarzając n-ty krok, możemy udostępnić modelowi pewną reprezentację historii/pamięci/stanu która w kompaktowy sposób przechowuje informację o "wszystkim co działo się wcześniej". W ten sposób zadania upraszcza sobie często np. modelowanie procesami stochastycznymi (założenie własności Markowa w modelu). W kontekście sieci neuronowych, odpowiednikiem takiego podejścia jest właśnie warstwa rekurencyjna: taka, która w n-tym kroku przetwarza n-te wejście i reprezentacje ukrytą z kroku n-1 (historię/stan/pamięć).

Oczywiście z góry warstwo wspomnieć o fakcie, że architektury rekurencyjne obecnie nie są pierwszym wyborem, w pracach state=of-the-art dominuje mechanizm uwagi. Transformery to obecnie podstawa wszystkiego, co kojarzone z sztuczną inteligencją - w szczególności wszelkiej maści "czatów" opartych o LLM (Large Language Models). Ale ich historia to w pierwszej kolejności dodanie mechanizmu uwagi do sieci rekurencyjnych, a dopiero potem spostrzeżenie, że po dodaniu tego mechanizmu, rekurencja nie jest już w zasadzie konieczna (bardzo znana publikacja *Attention Is All You Need*).

 Na tych laboratoriach przyjrzyjmy się warstwom LSTM - najpopularniejszej wersji architektury czysto rekurencyjnej.

# Zbiory danych

Przeprowadzimy test na dwóch zbiorach danych: klasyczny zbiór do analizy sentymentu tekstów (recenzje IMDB), oraz zbiór audio - rozpoznawanie owadów po wydawanych rzez nich odgłosach.

In [ ]:
import ssl
import certifi
from tensorflow.keras.datasets import imdb

ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())

(x_train, y_train), (x_test, y_test) = imdb.load_data(
    path='imdb.npz',
    num_words=None,
    skip_top=0,
    maxlen=None,
    seed=113,
    start_char=1,
    oov_char=2,
    index_from=3
)

In [ ]:
print(x_train[0])

In [ ]:
!pip install aeon

In [ ]:
from aeon.datasets import load_classification
X, y = load_classification("InsectWingbeat", extract_path="insect_data")

In [ ]:
print(X[0])

# Wczytywanie danych sekwencyjnych

Dla danych sekwencyjnych, przy przetwarzaniu w batchu pojawia się nowa komplikacja: batch musi być możliwy do "spakowania" w tensorze `batch_size x vector_dimension x sequence_length `, podczas gdy sekwencje w zbiorze uczącym nie muszą być jednakowego rozmiaru. Rozwiązaniem jest padding: dopełnianie tensorów zerami do stałej długości.

W problemach klasyfikacyjnych to powoduje jednak nowy problem: chcemy uzyskać reprezentację do klasyfikacji na końcu właściwej sekwencji, nie po przetworzeniu kilu (nastu/dziesięciu/set) dodatkowych wektorów zer. Implementacje warstw rekurencyjnych w torchu oferują nam rozwiązanie w postaci obiektów PackedSequence. Odpowiednimi funkcjami możemy "spakować" zarówno listę sekencji, jak i tensor już wypadowanych danych z podanymi osobno długościami sekwencji. Podanie takiej paczki na wejści warstwy rekurencyjnej gwarantuje, że warstwa zwróci nam swoją reprezentację na poziomie **ostatniego elementu właściwej sekwencji** (nie biorąc pod uwagę paddingu).

In [ ]:
import torch

sequences = [[0,1,2,3],
             [0,1],
             [0]]

packed_sequences = torch.nn.utils.rnn.pack_sequence([torch.tensor(seq) for seq in sequences])
print(packed_sequences)

padded_sequences = torch.nn.utils.rnn.pad_sequence([torch.tensor(seq) for seq in sequences])
print(padded_sequences)

packed_padded_sequences = torch.nn.utils.rnn.pack_padded_sequence(padded_sequences, [len(s) for s in sequences])
print(packed_padded_sequences)

# Zadanie 1

Zaimplementuj obiekty Dataset i DataLoader dla naszych zbiorów danych w wariantach zwracających zarówno obiekt PackedSequences, jak i prosty tensor z  wypadowanym zerami batchem.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

MAX_LEN    = 500          # przycinamy sekwencje do 500 tokenów
VOCAB_SIZE = max(max(s) for s in x_train) + 1  # maks indeks w danych + 1


In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pack_sequence

class IMDBDataset(Dataset):
    def __init__(self, sequences, labels, max_len=None):
        self.labels = torch.tensor(labels, dtype=torch.float32)
        if max_len:
            sequences = [s[:max_len] for s in sequences]
        self.sequences = [torch.tensor(s, dtype=torch.long) for s in sequences]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]


def collate_padded(batch):
    """Zwraca tensor z paddingiem do najdłuższej sekwencji w batchu."""
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs])
    padded = pad_sequence(seqs, batch_first=True, padding_value=0)
    return padded, lengths, torch.stack(labels)


def collate_packed(batch):
    """Zwraca PackedSequence posortowane malejąco po długości."""
    seqs, labels = zip(*batch)
    indices = sorted(range(len(seqs)), key=lambda i: len(seqs[i]), reverse=True)
    seqs_sorted   = [seqs[i]   for i in indices]
    labels_sorted = [labels[i] for i in indices]
    packed = pack_sequence(seqs_sorted)
    return packed, torch.stack(labels_sorted)


# Tworzymy datasety
train_ds = IMDBDataset(x_train, y_train, max_len=MAX_LEN)
test_ds  = IMDBDataset(x_test,  y_test,  max_len=MAX_LEN)

# DataLoadery — wariant z paddingiem
train_loader_pad = DataLoader(train_ds, batch_size=64, shuffle=True,  collate_fn=collate_padded)
test_loader_pad  = DataLoader(test_ds,  batch_size=64, shuffle=False, collate_fn=collate_padded)

# DataLoadery — wariant z PackedSequence
train_loader_pack = DataLoader(train_ds, batch_size=64, shuffle=True,  collate_fn=collate_packed)
test_loader_pack  = DataLoader(test_ds,  batch_size=64, shuffle=False, collate_fn=collate_packed)

# Szybki test
batch_pad, lengths, labels = next(iter(train_loader_pad))
print(f"[Padded]  batch shape: {batch_pad.shape}, lengths: {lengths[:5]}")

batch_pack, labels_pack = next(iter(train_loader_pack))
print(f"[Packed]  {batch_pack}")


# Embedding

Warstwy embeddingu są elementem wykorzystywanym przy przetwarzaniu danych, gdzie elementem jest id w pewnym dyskretnym zbiorze obiektów. Przykładowo, dla danych językowych może być to zbiór możliwych słów. Warstwa przyporządkowuje każdemu id jego własny wektor, i zakładamy, że w trakcie uczenia wyuczy się podobieństwa między obiektami.

## Model LSTM

W teorii, warstwy LSTM potrafią zapamiętywać "długoterminowo" a więc to skąd wyciągamy dane nie powinno robić większego problemu. Sprawdźmy czy tak jest w rzeczywistości, implementując wersję prostego eksperymentu - implementując uczenie zarówno z wykorzystaniem obiektów PackedSequence, jak i zwyczajnych tensorów dopełnionych zerami.

Nasz model LSTM musi być przygotowany na wszystkie opisane wersje naszego eksperymentu: dane w postaci sekwencji już w przestrzeni cech i w postaci sekwencji Integerów z embeddingiem w obrębie sieci; wykorzystanie PacekdSequences lub nie.

(**UWAGA:** W przypadku wykorzystania Embeddingu i Packed Sequences jednocześnie, będzie trzeba rozpakować, embedować i spakować jeszcze raz. Obsługiwanie tych czterech wariantów w jednej klasie nie jest praktyczne, potraktuj je raczej jako ćwiczenie.)

In [ ]:
class LSTMNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=1,
                 num_classes=1, bidirectional=False,
                 embed=False, vocab_size=None, packed=False):
        super().__init__()
        self.embed   = embed
        self.packed  = packed
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.bidirectional = bidirectional
        directions = 2 if bidirectional else 1

        if embed:
            assert vocab_size is not None
            self.embedding = nn.Embedding(vocab_size, input_size, padding_idx=0)

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional
        )
        self.fc = nn.Linear(hidden_size * directions, num_classes)

    def forward(self, x, lengths=None):
        if self.embed:
            if self.packed:
                # x to PackedSequence — trzeba rozpakować, embedować, spakować
                x_unpacked, lens = torch.nn.utils.rnn.pad_packed_sequence(x, batch_first=True)
                x_emb = self.embedding(x_unpacked)
                x = pack_padded_sequence(x_emb, lens.cpu(), batch_first=True, enforce_sorted=False)
            else:
                x = self.embedding(x)

        if self.packed:
            out, (h_n, _) = self.lstm(x)
        else:
            out, (h_n, _) = self.lstm(x)

        # h_n: (num_layers * directions, batch, hidden)
        if self.bidirectional:
            # bierzemy ostatnią warstwę — forward i backward
            h = torch.cat([h_n[-2], h_n[-1]], dim=1)
        else:
            h = h_n[-1]

        return self.fc(h).squeeze(-1)


# Szybki test modelu
model_test = LSTMNet(input_size=64, hidden_size=128, embed=True, vocab_size=VOCAB_SIZE)
batch_pad, lengths, labels = next(iter(train_loader_pad))
out = model_test(batch_pad)
print(f"Output shape: {out.shape}, przykładowe wartości: {out[:5].detach()}")

## Uczenie

Pętla ucząca dla modelu LSTM będzie analogiczna do znanych nam wcześniej

In [ ]:
def train_model(model, train_loader, test_loader, epochs=5, lr=1e-3,
                use_packed=False, device='cpu'):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    history = {'train_loss': [], 'train_acc': [], 'test_acc': []}

    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0

        for batch in train_loader:
            if use_packed:
                x, y = batch
                x = x.to(device)
            else:
                x, lengths, y = batch
                x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item() * y.size(0)
            correct    += ((out > 0) == y.bool()).sum().item()
            total      += y.size(0)

        train_loss = total_loss / total
        train_acc  = correct / total

        # Ewaluacja
        model.eval()
        correct_t, total_t = 0, 0
        with torch.no_grad():
            for batch in test_loader:
                if use_packed:
                    x, y = batch
                    x = x.to(device)
                else:
                    x, lengths, y = batch
                    x = x.to(device)
                y = y.to(device)
                out = model(x)
                correct_t += ((out > 0) == y.bool()).sum().item()
                total_t   += y.size(0)

        test_acc = correct_t / total_t
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)

        print(f"Epoch {epoch+1}/{epochs} | loss: {train_loss:.4f} | "
              f"train acc: {train_acc:.3f} | test acc: {test_acc:.3f}")

    return history

## Zadanie 2

Przeprowadź uczenie modelu LSTM na zadanym zbiorze i porównaj następujące podejścia:

*   LSTM, dane z paddingiem do długości najdłuższej sekwencji w batchu
*   Jak wyżej, ale Bidirectional
*   LSTM (nie bidirectional) z wykorzystaniem obiektów PaddedSequences

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EPOCHS = 5
HIDDEN = 128
EMBED_DIM = 64

results = {}

# --- Wariant 1: LSTM z paddingiem ---
print("=" * 50)
print("1. LSTM z paddingiem")
m1 = LSTMNet(input_size=EMBED_DIM, hidden_size=HIDDEN,
             embed=True, vocab_size=VOCAB_SIZE)
results['LSTM (padded)'] = train_model(
    m1, train_loader_pad, test_loader_pad,
    epochs=EPOCHS, use_packed=False, device=DEVICE
)

# --- Wariant 2: Bidirectional LSTM z paddingiem ---
print("=" * 50)
print("2. Bidirectional LSTM z paddingiem")
m2 = LSTMNet(input_size=EMBED_DIM, hidden_size=HIDDEN,
             embed=True, vocab_size=VOCAB_SIZE, bidirectional=True)
results['BiLSTM (padded)'] = train_model(
    m2, train_loader_pad, test_loader_pad,
    epochs=EPOCHS, use_packed=False, device=DEVICE
)

# --- Wariant 3: LSTM z PackedSequence ---
print("=" * 50)
print("3. LSTM z PackedSequence")
m3 = LSTMNet(input_size=EMBED_DIM, hidden_size=HIDDEN,
             embed=True, vocab_size=VOCAB_SIZE, packed=True)
results['LSTM (packed)'] = train_model(
    m3, train_loader_pack, test_loader_pack,
    epochs=EPOCHS, use_packed=True, device=DEVICE
)

# --- Wizualizacja wyników ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['steelblue', 'tomato', 'seagreen']
xs = range(1, EPOCHS + 1)

for (name, hist), color in zip(results.items(), colors):
    axes[0].plot(xs, hist['train_loss'], label=name, color=color, marker='o')
    axes[1].plot(xs, hist['test_acc'],  label=name, color=color, marker='o')

axes[0].set_title('Strata treningowa')
axes[0].set_xlabel('Epoka')
axes[0].set_ylabel('BCE Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].set_title('Dokładność na zbiorze testowym')
axes[1].set_xlabel('Epoka')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0.5, 1.0)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Porównanie wariantów LSTM — IMDB Sentiment', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Tabela końcowych wyników
print("\nTest accuracies:")
for name, hist in results.items():
    print(f"  {name:25s}: {hist['test_acc'][-1]:.3f}")


# Zadanie 3

Dla modelu językowego, dokonaj wizualizacji embeddingów 10 przykładowych słów. Dobierz słowa samodzielnie tak, aby pokazać, że embedding częściowo (ale prwadopodobnie nie idealnie) oddają relacje semantyczne. Możesz wykorzystać gotowe metody redukcji wymiarowości np. z scikit-learn aby umieścić embeddingi w przestrzeni dwuwymiarowej.

Ponieważ zbiór pobierany z tf.keras jest już reprezentowany w postaci list liczb całkowitych, bedziesz korzystać ze słownika indeksów również dostępnego w tf.keras:

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

# Słownik IMDB (indeks przesuwa się o 3 z powodu start_char, oov itd.)
word_idx = imdb.get_word_index(path='imdb_word_index.json')
idx_to_word = {v + 3: k for k, v in word_idx.items()}

# Słowa dobrane tak, żeby pokazać relacje semantyczne
WORDS = [
    "good", "bad", "great", "terrible",
    "film", "movie", "cinema",
    "love", "hate", "boring"
]

# Pobieramy embeddingi z wytrenowanego modelu (m1)
embedding_matrix = m1.embedding.weight.detach().cpu().numpy()

# Sprawdzamy indeksy (imdb index_from=3, stąd +3)
word_indices = {w: word_idx[w] + 3 for w in WORDS if w in word_idx}
valid_words  = [w for w in WORDS if w in word_indices]
vecs = np.array([embedding_matrix[word_indices[w]] for w in valid_words])

# Redukcja PCA do 2D
pca = PCA(n_components=2)
vecs_2d = pca.fit_transform(normalize(vecs))

# Wykres
fig, ax = plt.subplots(figsize=(9, 7))
colors_emb = plt.cm.tab10(np.linspace(0, 1, len(valid_words)))

for i, (word, (x, y)) in enumerate(zip(valid_words, vecs_2d)):
    ax.scatter(x, y, color=colors_emb[i], s=120, zorder=3)
    ax.annotate(word, (x, y), textcoords='offset points',
                xytext=(8, 4), fontsize=12, color=colors_emb[i], fontweight='bold')

ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.5, linestyle='--')
ax.set_title('Wizualizacja embeddingów (PCA 2D) — IMDB LSTM', fontsize=13, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% wariancji)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% wariancji)')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# Macierz podobieństwa kosinusowego
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix = cosine_similarity(normalize(vecs))

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(sim_matrix, cmap='RdYlGn', vmin=-1, vmax=1)
ax.set_xticks(range(len(valid_words))); ax.set_xticklabels(valid_words, rotation=45, ha='right')
ax.set_yticks(range(len(valid_words))); ax.set_yticklabels(valid_words)
plt.colorbar(im, ax=ax, label='Podobieństwo kosinusowe')
ax.set_title('Macierz podobieństwa embeddingów', fontsize=13, fontweight='bold')
for i in range(len(valid_words)):
    for j in range(len(valid_words)):
        ax.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center',
                fontsize=9, color='black')
plt.tight_layout()
plt.show()

In [ ]:
word_idx = imdb.get_word_index(
    path='imdb_word_index.json'
)
print(word_idx["good"])